In [2]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import os

In [3]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "v33.db" AS v33')

('ill', 'sisseütlev', 'illative'),

('in', 'seesütlev', 'inessive'),

('el', 'seestütlev', 'elative'),

('all', 'alaleütlev', 'allative'),

('ad', 'alalütlev', 'adessive'),

('abl', 'alaltütlev', 'ablative'),

## 1. tabel 
verb -> palju esineb obl+kääne (6 kohakäänet) : mitu distinct root 


In [5]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
"""

source = pd.read_sql_query(query, con)
source

,verb,verb_compound,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,12olema,,1,0,0,0,0,0,0
1,A. tihkama,,1,0,0,0,0,0,0
2,J. teppima,,1,0,0,0,0,0,0
3,Liitootama,,1,0,0,1,0,0,0
4,Southolema,,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
5651,ütlema,ära,5,2,138,67,488,3,54
5652,üürima,,69,18,107,67,49,9,153
5653,üürima,kokku,1,0,0,0,0,0,0
5654,šantažeerima,välja,1,0,0,0,0,0,0


# vahetabel

transactions_verbs_obl_kohakaandes_wcomps tabelile kus on verb+comp+kääne+elus+koht

juurde lilsada kas on alati/vahel/mitte kunagi/UNK isik

In [8]:
query = """
--
SELECT *
from 
transactions_verbs_obl_kohakaandes_wcomps
--transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
--transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps
"""

source = pd.read_sql_query(query, con)
source

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,kaane
0,2,toimuma,,1,lõpp,obl,S,"com,in,sg",UNK,UNK,in
1,3,saama,pihta,7,keel,obl,S,"all,com,pl",UNK,UNK,all
2,10,tulema,,19,sina,obl,P,"ad,sg",UNK,YES,ad
3,11,viilima,,22,tund,obl,S,"com,el,pl",UNK,UNK,el
4,11,viilima,,23,juht,obl,S,"ad,com,sg",UNK,YES,ad
...,...,...,...,...,...,...,...,...,...,...,...
7836717,30078975,minema,,54050467,perse,obl,S,"adit,com,sg",UNK,UNK,adit
7836718,30078981,rääkima,,54050472,sina,obl,P,"all,sg",UNK,YES,all
7836719,30078981,rääkima,,54050473,tege,obl,S,"abl,com,sg",UNK,UNK,abl
7836720,30078982,minema,,54050478,tyll,obl,S,"adit,com,sg",UNK,UNK,adit


In [6]:
# lahti pakitud märgenduste andmed
root = ".../margendatud"
margendused = pd.read_csv(os.path.join(root,"every_verb_case_obl.csv"), sep=";", encoding="utf-8")
margendused

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
0,saama - abl (kellelt/millelt),saama,abl (kellelt/millelt),vahel,vahel,mitte kunagi
1,tulema - abl (kellelt/millelt),tulema,abl (kellelt/millelt),vahel,vahel,mitte kunagi
2,küsima - abl (kellelt/millelt),küsima,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
3,nõudma - abl (kellelt/millelt),nõudma,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
4,võtma - abl (kellelt/millelt),võtma,abl (kellelt/millelt),vahel,vahel,mitte kunagi
...,...,...,...,...,...,...
10574,musitseerima - in (kelles/milles),musitseerima,in (kelles/milles),mitte kunagi,alati,muu
10575,kõigutama - in (kelles/milles),kõigutama,in (kelles/milles),mitte kunagi,mitte kunagi,muu
10576,kätlema - in (kelles/milles),kätlema,in (kelles/milles),mitte kunagi,alati,muu
10577,kõmmutama - in (kelles/milles),kõmmutama,in (kelles/milles),mitte kunagi,alati,mitte kunagi


### juurde lisada veerg, kas on isik alati/vahel/mittekunagi/unk

In [9]:
paarid = []
paarid_dict = {}

for i in range(len(margendused)):
    pair = margendused.iloc[i]["verbobl"]
    
    elems = pair.split("-")
    case = elems[1].split("(")[0].strip()
    verbelems = elems[0].strip().split(" ")
    verb = verbelems[0].strip()
    if len(verbelems)==1:
        comp = ""
    elif len(verbelems)==2:
        comp = verbelems[1].strip()
    
    if (verb,comp, case) not in paarid_dict.keys():
        paarid.append((verb,comp, case))
        paarid_dict[(verb, comp,case)] = margendused.iloc[i]["isikumäärus"]
    
    #if verb=='saama' and case == 'abl':
    #    print(verbelems)
    #    print(pair, comp)
    #    print(margendused.iloc[i]["isikumäärus"])
        #print(paarid_dict)
        #break

    
paarid = list(set(paarid))    
print(paarid[:3])

[('nõudma', 'läbi', 'abl'), ('solvama', '', 'adit'), ('pildistama', '', 'el')]


In [10]:
for i in tqdm(range(len(source))):
    v_k = (source.iloc[i]["verb"],source.iloc[i]["verb_compound"], source.iloc[i]["kaane"])
    if v_k in paarid_dict.keys():
        source.at[i, "isik"] =paarid_dict[v_k]
    else:
        source.at[i, "isik"] ="UNK"

100%|███████████████████████████████████████████████| 7836722/7836722 [13:03<00:00, 10006.10it/s]


In [11]:
source

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,kaane,isik
0,2,toimuma,,1,lõpp,obl,S,"com,in,sg",UNK,UNK,in,mitte kunagi
1,3,saama,pihta,7,keel,obl,S,"all,com,pl",UNK,UNK,all,vahel
2,10,tulema,,19,sina,obl,P,"ad,sg",UNK,YES,ad,alati
3,11,viilima,,22,tund,obl,S,"com,el,pl",UNK,UNK,el,UNK
4,11,viilima,,23,juht,obl,S,"ad,com,sg",UNK,YES,ad,UNK
...,...,...,...,...,...,...,...,...,...,...,...,...
7836717,30078975,minema,,54050467,perse,obl,S,"adit,com,sg",UNK,UNK,adit,vahel
7836718,30078981,rääkima,,54050472,sina,obl,P,"all,sg",UNK,YES,all,vahel
7836719,30078981,rääkima,,54050473,tege,obl,S,"abl,com,sg",UNK,UNK,abl,mitte kunagi
7836720,30078982,minema,,54050478,tyll,obl,S,"adit,com,sg",UNK,UNK,adit,vahel


In [12]:
cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_wcomps_isik
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_wcomps_isik', con=con)

7836722

## millised verbid + kääne on juba märgendatud ja millist on veel vaja 

In [13]:
q = """
SELECT 
    distinct verb,
    verb_compound,
    kaane, 
    count(distinct root_word) as root_count,
    'false' as annotated

FROM transactions_verbs_obl_kohakaandes_wcomps
group by verb,verb_compound, kaane
order by root_count desc
"""

s3 = pd.read_sql_query(q, con)
s3

,verb,verb_compound,kaane,root_count,annotated
0,saama,,el,19632,false
1,andma,,all,11612,false
2,rääkima,,el,10891,false
3,saama,,in,8532,false
4,tulema,,ad,8468,false
...,...,...,...,...,...
74715,šveitsima,,el,1,false
74716,švipsima,,ad,1,false
74717,žestikuleerima,,ad,1,false
74718,žisraelima,,ad,1,false


In [14]:
for i in tqdm(range(len(s3))):
    v_k = (s3.iloc[i]["verb"],s3.iloc[i]["verb_compound"], s3.iloc[i]["kaane"])
    if v_k in paarid:
        s3.at[i, "annotated"] ='true'

100%|████████████████████████████████████████████████████| 74720/74720 [00:22<00:00, 3342.07it/s]


In [15]:
s3

,verb,verb_compound,kaane,root_count,annotated
0,saama,,el,19632,true
1,andma,,all,11612,true
2,rääkima,,el,10891,true
3,saama,,in,8532,true
4,tulema,,ad,8468,true
...,...,...,...,...,...
74715,šveitsima,,el,1,false
74716,švipsima,,ad,1,false
74717,žestikuleerima,,ad,1,false
74718,žisraelima,,ad,1,false


In [16]:
s3.to_csv("transactions_verbs_obl_kohakaandes_root_counts_distinct_coverage_v2_wcomps.csv", encoding="utf-8", index=False, sep=";")

In [19]:
s3.to_sql(name='transactions_verbs_obl_kohakaandes_root_counts_distinct_coverage_v2_wcomps', con=con)

74720

In [21]:
s3[(s3["annotated"]=='false')]

,index,verb,verb_compound,kaane,root_count,annotated
3437,3437,jooksma,ringi,in,99,false
3440,3440,kalduma,,el,99,false
3441,3441,kallinema,,el,99,false
3444,3444,klappima,,ad,99,false
3445,3445,klappima,,in,99,false
...,...,...,...,...,...,...
74715,74715,šveitsima,,el,1,false
74716,74716,švipsima,,ad,1,false
74717,74717,žestikuleerima,,ad,1,false
74718,74718,žisraelima,,ad,1,false


## 2. tabel

iga verb+obl+kohakääne jaoks count elus ja count koht, count kokku

 transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
 
 transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches

In [22]:
query = """
SELECT *
from
transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
"""

source = pd.read_sql_query(query, con)
source

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,saama,,el,1274,402,19632
1,andma,,all,1601,250,11612
2,rääkima,,el,802,202,10891
3,saama,,in,134,306,8532
4,tulema,,ad,1134,166,8468
...,...,...,...,...,...,...
74715,šveitsima,,el,0,0,1
74716,švipsima,,ad,0,0,1
74717,žestikuleerima,,ad,0,1,1
74718,žisraelima,,ad,0,0,1


### lisada juurde isiku staatus

In [23]:
for i in tqdm(range(len(source))):
    v_k = (source.iloc[i]["verb"],source.iloc[i]["verb_compound"], source.iloc[i]["kaane"])
    if v_k in paarid_dict.keys():
        source.at[i, "isik"] =paarid_dict[v_k]
    else:
        source.at[i, "isik"] ="UNK"

100%|███████████████████████████████████████████████████| 74720/74720 [00:07<00:00, 10530.94it/s]


In [24]:
source

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt,isik
0,saama,,el,1274,402,19632,vahel
1,andma,,all,1601,250,11612,vahel
2,rääkima,,el,802,202,10891,vahel
3,saama,,in,134,306,8532,mitte kunagi
4,tulema,,ad,1134,166,8468,alati
...,...,...,...,...,...,...,...
74715,šveitsima,,el,0,0,1,UNK
74716,švipsima,,ad,0,0,1,UNK
74717,žestikuleerima,,ad,0,1,1,UNK
74718,žisraelima,,ad,0,0,1,UNK


In [25]:
source.to_sql(name='transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1_isik', con=con)

74720

## kas elus:koht suhtel on mingi seos alati/vahel/mitte kunagi puhul

In [27]:
query = """
SELECT * from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1_isik
"""

source = pd.read_sql_query(query, con)

In [31]:
source["elus_suhe"] = 0

In [33]:
for i in tqdm(range(len(source))):
    suhe = source.iloc[i]["elus_cnt"]/source.iloc[i]["koht_cnt"]
    source.at[i, "elus_suhe"] = suhe

  0%|                                                                  | 0/74720 [00:00<?, ?it/s]/tmp/ipykernel_4847/337345088.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '3.1691542288557213' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  source.at[i, "elus_suhe"] = suhe
  2%|▉                                                   | 1273/74720 [00:00<00:05, 12725.04it/s]/tmp/ipykernel_4847/337345088.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  suhe = source.iloc[i]["elus_cnt"]/source.iloc[i]["koht_cnt"]
  3%|█▊                                                  | 2585/74720 [00:00<00:05, 12956.43it/s]/tmp/ipykernel_4847/337345088.py:2: RuntimeWarning: invalid value encountered in scalar divide
  suhe = source.iloc[i]["elus_cnt"]/source.iloc[i]["koht_cnt"]
100%|███████████████████████████████████████████████████| 74720/74720 [00:05<00:00, 12676.01it/s

In [34]:
source

,index,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt,isik,elus_suhe
0,0,saama,,el,1274,402,19632,vahel,3.169154
1,1,andma,,all,1601,250,11612,vahel,6.404000
2,2,rääkima,,el,802,202,10891,vahel,3.970297
3,3,saama,,in,134,306,8532,mitte kunagi,0.437908
4,4,tulema,,ad,1134,166,8468,alati,6.831325
...,...,...,...,...,...,...,...,...,...
74715,74715,šveitsima,,el,0,0,1,UNK,NaN
74716,74716,švipsima,,ad,0,0,1,UNK,NaN
74717,74717,žestikuleerima,,ad,0,1,1,UNK,0.000000
74718,74718,žisraelima,,ad,0,0,1,UNK,NaN


In [36]:
source.to_csv("transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1_isik_suhe.csv", sep=",", encoding="utf-8", index=False)

## elus vs koht verbid

### verbid mill epuhul kohtasid on rohkem kui elus

In [37]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
where koht_cnt > elus_cnt
order by koht_cnt desc
limit 500
"""

source2 = pd.read_sql_query(query, con)
source2

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,käima,,in,99,351,6832
1,elama,,in,90,333,4597
2,toimuma,,in,86,332,6732
3,asuma,,in,49,324,4179
4,saama,,in,134,306,8532
...,...,...,...,...,...,...
495,laskma,,ill,11,38,302
496,kukkuma,,adit,15,38,292
497,maanduma,,all,17,38,259
498,jooksma,,ill,12,38,253


### elus:koht suhe on 3:1

In [38]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
where elus_cnt > koht_cnt
and elus_cnt >= 3*koht_cnt
order by elus_cnt desc
limit 500
"""

source = pd.read_sql_query(query, con)
source

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,andma,,all,1601,250,11612
1,saama,,el,1274,402,19632
2,tegema,,all,1143,210,7258
3,tulema,,ad,1134,166,8468
4,pakkuma,,all,1016,113,4235
...,...,...,...,...,...,...
495,väärima,,el,36,8,250
496,puhuma,,all,36,9,147
497,ilmutama,,all,36,7,138
498,nõudma,tagasi,abl,36,6,118


### koht:elus suhe on 3:1

In [39]:
# koht

query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
where koht_cnt > elus_cnt
and koht_cnt >= 3*elus_cnt
order by koht_cnt desc
limit 500
"""

source2 = pd.read_sql_query(query, con)
source2




,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,käima,,in,99,351,6832
1,elama,,in,90,333,4597
2,toimuma,,in,86,332,6732
3,asuma,,in,49,324,4179
4,töötama,,in,67,220,5094
...,...,...,...,...,...,...
495,jõudma,välja,el,5,24,128
496,blokeerima,,in,3,24,120
497,avanema,,adit,3,24,102
498,jalutama,,adit,3,24,87


In [40]:
con.close()